In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


class Sampling(layers.Layer):

    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon



class VAE(keras.Model):
    def __init__(self, input_dim, **kwargs):
        super().__init__(**kwargs)

        self.latent_dim = 10
        self.input_dim = input_dim

        encoder_inputs = keras.Input(shape=(self.input_dim,))
        x = layers.Dense(16, activation="relu")(encoder_inputs)
        z_mean = layers.Dense(self.latent_dim, name="z_mean")(x)
        z_log_var = layers.Dense(self.latent_dim, name="z_log_var")(x)
        z = Sampling()([z_mean, z_log_var])
        self.encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")


        latent_inputs = keras.Input(shape=(self.latent_dim,))
        x = layers.Dense(16, activation="relu")(latent_inputs)
        x = layers.Dense(30, activation="relu")(x)
        decoder_outputs = layers.Dense(self.input_dim, activation="sigmoid")(x)
        self.decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")


        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [
            self.loss_tracker,
        ]
    
    def vae_loss(self, data):
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)
        reconstruction_loss = tf.reduce_mean(keras.losses.binary_crossentropy(data, reconstruction))

        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        total_loss = reconstruction_loss + kl_loss
        return total_loss, kl_loss, reconstruction_loss

    def train_step(self, data):
        with tf.GradientTape() as tape:
            total_loss = self.vae_loss(data)
        
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.loss_tracker.update_state(total_loss)

        return {
            "loss": self.loss_tracker.result(),
        }


input_data = np.random.random((100, 30))  # Example input data with shape (None, 30)


df = pd.read_csv('creditcard.csv')


target_feature = "Class"
# instantiate two more datasets one to contain the normal data
# and the other to hold the anomalous instances
normal_data = df[df[target_feature] == 0].drop(columns=target_feature)
anomalous_data = df[df[target_feature] == 1].drop(columns=target_feature)

# also instantiate a dataframe to carry all the features
# and a second one to instead contain the target variable
features = df.drop([target_feature], axis=1)
label = df[target_feature]

# the stratify parameter here is set to be "isghost" meaning that the
# proportions of ghost and normal tracks in both train and test datasets
# will be kept the same...
x_train, x_test, y_train, y_test = train_test_split(
    features, label, test_size=0.2, stratify=label
)

# remember that at least for the autoencoders we're gonna use just the
# normal instances during the training phase so we have to filter those in x_train
train_index = y_train[y_train == 0].index
train_normal_data = x_train.loc[train_index]

train_normal_data = (train_normal_data - train_normal_data.mean()) / train_normal_data.std()
x_test = (x_test - train_normal_data.mean()) / train_normal_data.std()

#now moving the data to a fixed range of values 
scaler = MinMaxScaler()

train_normal_data = scaler.fit_transform(train_normal_data)
x_test = scaler.fit_transform(x_test)


print("this is the shape of the data")

print(train_normal_data.shape)
print(train_normal_data.shape[1])

print(input_data.shape)

vae = VAE(input_dim=train_normal_data.shape[1])
vae.compile(optimizer=keras.optimizers.Adam())
vae.fit(train_normal_data, epochs=3, batch_size=128)


this is the shape of the data
(227451, 30)
30
(100, 30)
Epoch 1/3
1777/1777 [==============================] - 3s 2ms/step - loss: 0.4120
Epoch 2/3
 252/1777 [===>..........................] - ETA: 2s - loss: 0.4066

In [23]:
type(input_data)

numpy.ndarray

In [25]:
type(train_normal_data)

numpy.ndarray

In [26]:
train_normal_data

array([[ 4.20430000e+04, -2.81297399e-01,  7.18910284e-01, ...,
         2.16621016e-01,  2.42395609e-01,  3.74500000e+01],
       [ 7.52990000e+04, -6.74605206e-01,  1.39288946e+00, ...,
         1.21351344e-01,  3.53212707e-02,  8.90000000e-01],
       [ 1.51390000e+05,  1.83226470e+00,  1.58658366e-01, ...,
        -5.26252485e-02, -5.67721139e-02,  2.35900000e+01],
       ...,
       [ 1.67403000e+05, -7.15311477e-01,  1.67339230e+00, ...,
         7.28391572e-01,  4.79967305e-01,  5.85000000e+00],
       [ 9.63640000e+04, -8.26148878e-01,  6.87366988e-01, ...,
         2.18519998e-01,  1.19809978e-01,  2.99900000e+01],
       [ 5.49020000e+04, -1.48032395e+00, -7.21491171e-01, ...,
        -7.81556366e-02, -1.17713050e-01,  7.60000000e-01]])

In [27]:
input_data

array([[0.34188726, 0.11179247, 0.55456561, ..., 0.99050813, 0.40725489,
        0.95779918],
       [0.50073297, 0.54129411, 0.56626755, ..., 0.13058545, 0.1120663 ,
        0.69351546],
       [0.06872465, 0.12488304, 0.39178921, ..., 0.20525197, 0.40053285,
        0.05766611],
       ...,
       [0.43274769, 0.65475035, 0.05569425, ..., 0.29619877, 0.06700489,
        0.11244236],
       [0.65791314, 0.92426111, 0.69328331, ..., 0.74527545, 0.32707887,
        0.06572791],
       [0.1722951 , 0.24671258, 0.1931012 , ..., 0.39527871, 0.72776207,
        0.24178701]])